# Harmonic Analysis and THD

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from power_electronics.converters.dc_ac.single_phase_hbridge import SinglePhaseHBridgeVSI
from power_electronics.converters.dc_ac.three_phase_vsi import ThreePhaseVSI
from power_electronics.converters.ac_dc.full_bridge import FullBridgeRectifier
from power_electronics.core.analysis import compute_fft, compute_thd

In [ ]:
models={'hbridge':SinglePhaseHBridgeVSI({'Vdc':300,'f_output':50,'f_carrier':5000,'modulation_index':0.8,'R_load':20,'L_load':0.01,'C_load':0.0,'control_mode':'SPWM'}),'three_phase_vsi':ThreePhaseVSI({'Vdc':600,'f_output':50,'f_carrier':8000,'modulation_index':0.9,'R_load':15,'L_load':0.02,'C_load':0.0,'control_mode':'SPWM'}),'full_bridge_rectifier':FullBridgeRectifier({'Vac_peak':170,'frequency':50,'R_load':20,'L_filter':1e-3,'C_filter':2e-3})}
rows=[]
for name,m in models.items():
    r=m.simulate(0.1,5000); v=r.signals.get('Vout',r.signals.get('Vdc_filtered'))
    fs=1/(r.time[1]-r.time[0]); thd=compute_thd(v if getattr(v,'ndim',1)==1 else v[:,0],50,fs)
    rows.append({'converter':name,'THD_%':thd,'power_factor':r.metrics.get('power_factor',np.nan)})
pd.DataFrame(rows)

In [ ]:
caps=np.linspace(100e-6,5e-3,25); thds=[]
for c in caps:
    r=FullBridgeRectifier({'Vac_peak':170,'frequency':50,'R_load':20,'L_filter':1e-3,'C_filter':float(c)}).simulate(0.15,5000)
    fs=1/(r.time[1]-r.time[0]); thds.append(compute_thd(r.signals['Vdc_filtered'],100,fs))
plt.figure(figsize=(7,3)); plt.plot(caps*1e6,thds); plt.xlabel('C (uF)'); plt.ylabel('THD (%)'); plt.grid(True,alpha=0.3)